# Cyber-LLM 7B QLoRA Training on Kaggle (Free T4×2 = 32GB VRAM)

**Instructions:**
1. Upload this notebook to Kaggle (kaggle.com/code → New Notebook)
2. Settings → Accelerator → **GPU T4 x2** (32GB VRAM)
3. Settings → Internet → **On**
4. Add Secrets (🔑 icon): `WANDB_API_KEY`, `HF_TOKEN`
5. Run all cells sequentially

**Expected time:** ~2-3 hours on T4×2 (32GB VRAM)
**Output:** 7B GGUF model in `/kaggle/working/qwen25-coder-7b-cyber/gguf/`

In [ ]:
# Cell 0: Memory optimization config (MUST RUN FIRST)
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch
torch.cuda.empty_cache()

# Cell 1: Clone repository and install dependencies
!wget -q https://codeload.github.com/1stsonushaw4590-wq/Own-AI/zip/refs/heads/master -O Own-AI.zip
!unzip -q Own-AI.zip
!mv Own-AI-master Own-AI
%cd Own-AI

# Install dependencies
!pip install -q -r training/requirements.txt
!pip install -q bitsandbytes accelerate peft trl wandb huggingface_hub

# Verify GPU
import torch
print(f"CUDA: {torch.cuda.is_available()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} ({torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB)")

In [ ]:
# Cell 2: Configure secrets
import os

# Add these as Kaggle Secrets (🔑 icon in left sidebar):
#   WANDB_API_KEY = your_wandb_key
#   HF_TOKEN = your_hf_token

os.environ["WANDB_API_KEY"] = os.getenv("WANDB_API_KEY", "")
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN", "")

print(f"WANDB: {'✅' if os.getenv('WANDB_API_KEY') else '❌'}")
print(f"HF_TOKEN: {'✅' if os.getenv('HF_TOKEN') else '❌'}")

In [ ]:
# Cell 3: Verify dataset
import json, os

DATASET_PATH = "data/cyber_train_merged.jsonl"
if os.path.exists(DATASET_PATH):
    with open(DATASET_PATH) as f:
        lines = f.readlines()
    print(f"Dataset: {len(lines)} samples")
    sample = json.loads(open(DATASET_PATH).readline())
    print(f"Keys: {list(json.loads(open(DATASET_PATH).readline()).keys())}")
else:
    print("Dataset not found - run from repo root")
    !ls -la data/

In [ ]:
# Cell 4: Training configuration
import torch
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, TrainingArguments, Trainer
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import load_dataset
from transformers import DataCollatorForSeq2Seq
import wandb

MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"
DATASET_PATH = "data/cyber_train_merged.jsonl"
OUTPUT_DIR = "/kaggle/working/qwen25-coder-7b-cyber"
MAX_SEQ_LENGTH = 1024
BATCH_SIZE = 1
GRAD_ACCUM = 16
LEARNING_RATE = 2e-4
NUM_EPOCHS = 3
MAX_SAMPLES = 0

QLORA_CONFIG = {
    "load_in_4bit": True,
    "bnb_4bit_compute_dtype": torch.bfloat16,
    "bnb_4bit_use_double_quant": True,
    "bnb_4bit_quant_type": "nf4",
}

LORA_CONFIG = LoraConfig(
    r=64,
    lora_alpha=128,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

TRAINING_CONFIG = {
    "output_dir": OUTPUT_DIR,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 16,
    "num_train_epochs": 3,
    "learning_rate": 2e-4,
    "lr_scheduler_type": "cosine",
    "warmup_ratio": 0.03,
    "logging_steps": 10,
    "save_steps": 500,
    "save_total_limit": 3,
    "bf16": True,
    "tf32": True,
    "gradient_checkpointing": True,
    "gradient_checkpointing_kwargs": {"use_reentrant": False},
    "optim": "paged_adamw_8bit",
    "max_grad_norm": 0.3,
    "max_seq_length": 1024,
    "packing": False,
    "report_to": "wandb" if os.getenv("WANDB_API_KEY") else "none",
    "dataloader_pin_memory": False,
    "dataloader_num_workers": 2,
    "remove_unused_columns": False,
    "dataloader_drop_last": True,
    "max_memory": {0: "14GB", 1: "14GB"},
    "device_map": "auto"
}

In [ ]:
# Cell 5: Training function
import json
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM,
    BitsAndBytesConfig, TrainingArguments, Trainer,
    DataCollatorForSeq2Seq
)
from peft import (
    LoraConfig, get_peft_model,
    prepare_model_for_kbit_training, PeftModel
)
from trl import SFTTrainer
import wandb

# Qwen chat template
QWEN_CHAT_TEMPLATE = """{% for message in messages %}{% if message['role'] == 'system' %}{{ '\u7cfb统\n' + message['content'] + ' \u4ed8底' }}{% elif message['role'] == 'user' %}{{ '\u7528户\n' + message['content'] + ' \u4ed8底' }}{% elif message['role'] == 'assistant' %}{{ '\u52a9手\n' + message['content'] + ' \u4ed8底' }}{% endif %}{% endfor %}{% if add_generation_prompt %}{{ '\u52a9手\n' }}{% endif %}"""

def format_chat(example):
    if "messages" in example:
        messages = example["messages"]
    elif "instruction" in example and "output" in example:
        messages = [
            {"role": "user", "content": example["instruction"]},
            {"role": "assistant", "content": example["output"]}
        ]
    else:
        keys = list(example.keys())
        if len(keys) >= 2:
            messages = [
                {"role": "user", "content": str(example[list(example.keys())[0]])},
                {"role": "assistant", "content": str(example[list(example.keys())[1]])}
            ]
        else:
            messages = [{"role": "user", "content": str(example)}]
    
    # Ensure messages is a valid list
    if not isinstance(messages, list) or len(messages) == 0:
        messages = [{"role": "user", "content": str(example)}]
    
    # Validate each message has role and content
    valid_messages = []
    for msg in messages:
        if isinstance(msg, dict) and "role" in msg and "content" in msg:
            role = msg["role"]
            content = str(msg["content"]) if msg["content"] is not None else ""
            if role in ["user", "assistant", "system"] and content.strip():
                valid_messages.append({"role": role, "content": content})
    
    if not valid_messages:
        valid_messages = [{"role": "user", "content": "Hello"}]
    
    text = tokenizer.apply_chat_template(
        valid_messages, tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

def train():
    global tokenizer
    
    if os.getenv("WANDB_API_KEY"):
        wandb.init(project="cyber-llm-7b", config={
            "model": "Qwen2.5-Coder-7B",
            "dataset": "cyber-llm-merged",
            "epochs": NUM_EPOCHS,
        })
    
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME, trust_remote_code=True, padding_side="right"
    )
    tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    tokenizer.model_max_length = 2048

    # Set Qwen chat template
    tokenizer.chat_template = QWEN_CHAT_TEMPLATE

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
        dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
        max_memory={0: "14GB", 1: "14GB"},
    )

    model = prepare_model_for_kbit_training(model)
    model.config.use_cache = False
    model.config.pretraining_tp = 1

    lora_config = LoraConfig(
        r=64, lora_alpha=128,
        target_modules=["q_proj","k_proj","v_proj","o_proj",
                       "gate_proj","up_proj","down_proj"],
        lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
    )
    model = get_peft_model(model, LoraConfig(
        r=64, lora_alpha=128,
        target_modules=["q_proj","k_proj","v_proj","o_proj",
                       "gate_proj","up_proj","down_proj"],
        lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
    ))
    model.print_trainable_parameters()

    dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
    if MAX_SAMPLES > 0 and len(dataset) > MAX_SAMPLES:
        dataset = dataset.select(range(MAX_SAMPLES))
    print(f"Dataset: {len(dataset)} samples")

    def format_chat(example):
        if "messages" in example:
            messages = example["messages"]
        elif "instruction" in example and "output" in example:
            messages = [
                {"role": "user", "content": example["instruction"]},
                {"role": "assistant", "content": example["output"]}
            ]
        else:
            keys = list(example.keys())
            if len(keys) >= 2:
                messages = [
                    {"role": "user", "content": str(example[list(example.keys())[0]])},
                    {"role": "assistant", "content": str(example[list(example.keys())[1]])}
                ]
            else:
                messages = [{"role": "user", "content": str(example)}]
        
        # Ensure messages is a valid list
        if not isinstance(messages, list) or len(messages) == 0:
            messages = [{"role": "user", "content": str(example)}]
        
        # Validate each message has role and content
        valid_messages = []
        for msg in messages:
            if isinstance(msg, dict) and "role" in msg and "content" in msg:
                role = msg["role"]
                content = str(msg["content"]) if msg["content"] is not None else ""
                if role in ["user", "assistant", "system"] and content.strip():
                    valid_messages.append({"role": role, "content": content})
    
    if not valid_messages:
        valid_messages = [{"role": "user", "content": "Hello"}]
    
    text = tokenizer.apply_chat_template(
        valid_messages, tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

    dataset = dataset.map(format_chat, remove_columns=dataset.column_names)
    print(f"Formatted {len(dataset)} samples")

    def tokenize_fn(examples):
        tok = tokenizer(
            examples["text"],
            truncation=True, padding="max_length",
            max_length=MAX_SEQ_LENGTH,
        )
        tok["labels"] = tok["input_ids"].copy()
        return tok

    dataset = dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=["text"],
    num_proc=2,
    )

    dataset = dataset.train_test_split(test_size=0.02, seed=42)
    print(f"Train: {len(dataset['train'])}, Eval: {len(dataset['test'])}")

    data_collator = DataCollatorForSeq2Seq(
        tokenizer=tokenizer, model=model, padding=True, label_pad_token_id=-100
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        args=TrainingArguments(**TRAINING_CONFIG),
        train_dataset=dataset["train"],
        eval_dataset=dataset["test"],
        max_seq_length=MAX_SEQ_LENGTH,
        dataset_text_field="text",
data_collator=data_collator,
packing=False,
    )

    print("Starting training...")
    print(f"  Model: {MODEL_NAME}")
    print(f"  Output: {OUTPUT_DIR}")
    print(f"  Epochs: {NUM_EPOCHS}")
    print(f"  Batch size: {BATCH_SIZE} (grad accum: {GRAD_ACCUM})")
    print(f"  LR: {LEARNING_RATE}")
    print(f"  Max seq len: {MAX_SEQ_LENGTH}")

    trainer.train()

    print(f"\nSaving adapter to {OUTPUT_DIR}/adapter")
    trainer.save_model(f"{OUTPUT_DIR}/adapter")
    tokenizer.save_pretrained(f"{OUTPUT_DIR}/adapter")

    config = {
        "base_model": MODEL_NAME,
        "qlora": QLORA_CONFIG,
        "lora": {
            "r": 64, "lora_alpha": 128,
            "target_modules": ["q_proj","k_proj","v_proj","o_proj",
                           "gate_proj","up_proj","down_proj"],
            "lora_dropout": 0.05, "bias": "none", "task_type": "CAUSAL_LM"
        },
        "training": {k: v for k, v in TRAINING_CONFIG.items() if k != "report_to"},
        "dataset": DATASET_PATH,
        "max_samples": MAX_SAMPLES,
        "max_seq_length": MAX_SEQ_LENGTH,
    }
    with open(f"{OUTPUT_DIR}/training_config.json", "w") as f:
        json.dump(config, f, indent=2, default=str)

    print(f"\nTraining complete! Adapter saved to {OUTPUT_DIR}/adapter")
    
    if os.getenv("WANDB_API_KEY"):
        wandb.finish()

    return OUTPUT_DIR

In [ ]:
# Cell 6: Execute training
import os, torch
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    TrainingArguments, Trainer, DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import load_dataset
import wandb

MODEL_NAME = "Qwen/Qwen2.5-Coder-7B-Instruct"
DATASET_PATH = "data/cyber_train_merged.jsonl"
OUTPUT_DIR = "/kaggle/working/qwen25-coder-7b-cyber"
MAX_SEQ_LENGTH = 1024
BATCH_SIZE = 1
GRAD_ACCUM = 16
LEARNING_RATE = 2e-4
NUM_EPOCHS = 3
MAX_SAMPLES = 0

QLORA_CONFIG = {
    "load_in_4bit": True,
    "bnb_4bit_compute_dtype": torch.bfloat16,
    "bnb_4bit_use_double_quant": True,
    "bnb_4bit_quant_type": "nf4",
}

LORA_CONFIG = LoraConfig(
    r=64, lora_alpha=128,
    target_modules=["q_proj","k_proj","v_proj","o_proj",
                       "gate_proj","up_proj","down_proj"],
    lora_dropout=0.05, bias="none", task_type="CAUSAL_LM"
)

TRAINING_CONFIG = {
    "output_dir": OUTPUT_DIR,
    "per_device_train_batch_size": 1,
    "gradient_accumulation_steps": 16,
    "num_train_epochs": 3,
    "learning_rate": 2e-4,
    "lr_scheduler_type": "cosine",
    "warmup_ratio": 0.03,
    "logging_steps": 10,
    "save_steps": 500,
    "save_total_limit": 3,
    "bf16": True, "tf32": True,
    "gradient_checkpointing": True,
    "gradient_checkpointing_kwargs": {"use_reentrant": False},
    "optim": "paged_adamw_8bit",
    "max_grad_norm": 0.3,
    "max_seq_length": 1024,
    "packing": False,
    "report_to": "wandb" if os.getenv("WANDB_API_KEY") else "none",
    "dataloader_pin_memory": False,
    "dataloader_num_workers": 2,
    "remove_unused_columns": False,
    "dataloader_drop_last": True,
    "max_memory": {0: "14GB", 1: "14GB"},
    "device_map": "auto"
}

import os, torch
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    TrainingArguments, Trainer, DataCollatorForSeq2Seq
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer
from datasets import load_dataset
import wandb

os.makedirs(OUTPUT_DIR, exist_ok=True)
output_dir = train()
print(f"Training complete: {output_dir}")

In [ ]:
# Cell 7: Merge adapter and quantize to GGUF
!python3 training/merge_and_quantize.py \
  --base Qwen/Qwen2.5-Coder-7B-Instruct \
  --adapter /kaggle/working/qwen25-coder-7b-cyber/adapter \
  --output /kaggle/working/qwen25-coder-7b-cyber/merged \
  --gguf /kaggle/working/qwen25-coder-7b-cyber/gguf \
  --quant q4_k_m

In [ ]:
# Cell 8: Alternative - Manual merge if script fails
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import os

ADAPTER_PATH = "/kaggle/working/qwen25-coder-7b-cyber/adapter"
MERGED_DIR = "/kaggle/working/qwen25-coder-7b-cyber/merged"
GGUF_DIR = "/kaggle/working/qwen25-coder-7b-cyber/gguf"

os.makedirs(MERGED_DIR, exist_ok=True)
os.makedirs(GGUF_DIR, exist_ok=True)

print("Loading base model...")
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-7B-Instruct", trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-Coder-7B-Instruct",
    dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    max_memory={0: "14GB", 1: "14GB"},
)

print("Loading adapter...")
model = PeftModel.from_pretrained(model, ADAPTER_PATH)
print("Merging...")
model = model.merge_and_unload()

print("Saving merged model...")
model.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-Coder-7B-Instruct", trust_remote_code=True)
tokenizer.save_pretrained(MERGED_DIR)
print("Merge complete!")

# Note: For GGUF conversion, run locally:
# python3 /path/to/llama.cpp/convert_hf_to_gguf.py merged/ --outfile model-f16.gguf --outtype f16
# ./llama-quantize model-f16.gguf model-q4_k_m.gguf q4_k_m

In [ ]:
# Cell 9: Verify outputs and create summary
print("\n" + "="*60)
print("KAGGLE TRAINING COMPLETE - OUTPUT SUMMARY")
print("="*60)

outputs = [
    ("/kaggle/working/qwen25-coder-7b-cyber/adapter", "LoRA Adapter"),
    ("/kaggle/working/qwen25-coder-7b-cyber/merged", "Merged Model (FP16)"),
    ("/kaggle/working/qwen25-coder-7b-cyber/gguf/cyber-llm-q4_k_m.gguf", "GGUF Q4_K_M"),
    ("/kaggle/working/qwen25-coder-7b-cyber/gguf/cyber-llm-f16.gguf", "GGUF FP16"),
]

for path, desc in outputs:
    if os.path.exists(path):
        size = os.path.getsize(path) / (1024**3) if os.path.isfile(path) else sum(
            os.path.getsize(os.path.join(path, f)) for f in os.listdir(path)
        ) / (1024**3)
        print(f"OK {desc}: {path} ({size:.2f} GB)")
    else:
        print(f"MISSING {desc}: {path}")

print("\nNext steps:")
print("1. Download GGUF from Kaggle output (right-click > Download)")
print("2. Or push to HF Hub: huggingface-cli upload your-username/cyber-llm-7b /kaggle/working/qwen25-coder-7b-cyber/gguf")
print("3. Deploy locally: llama-server -m cyber-llm-q4_k_m.gguf --port 8080")
print("4. Test orchestrator: ./agent/cpp/build/cyber_orchestrator --prompt 'scan 127.0.0.1'")

if os.getenv("HF_TOKEN"):
    print("\nPushing to HF Hub...")
    !huggingface-cli upload your-username/cyber-llm-7b /kaggle/working/qwen25-coder-7b-cyber/gguf --repo-type model

In [ ]:
# Cell 10: Quick inference test (optional)
print("Testing inference on merged model...")

from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tokenizer = AutoTokenizer.from_pretrained(
    "/kaggle/working/qwen25-coder-7b-cyber/merged",
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    "/kaggle/working/qwen25-coder-7b-cyber/merged",
    dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True,
    max_memory={0: "14GB", 1: "14GB"},
)

def generate(prompt, max_new_tokens=200):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            temperature=0.2, do_sample=True, top_p=0.9
        )
    return tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print("Test 1: What is SQL injection?")
print(generate("What is SQL injection and how to prevent it?"))

print("\nTest 2: Nmap command")
print(generate("Show nmap command to scan 192.168.1.1"))